### Experiment Notebook
Day-ahead ERCOT South Central load forecasting — SHAP-guided feature engineering, reproducing the source paper's experiment sequence.


#### Environment setup
Load credentials from environment variables

In [8]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['MLFLOW_TRACKING_URI'] =os.getenv("MLFLOW_TRACKING_URI")
os.environ['MLFLOW_TRACKING_USERNAME'] = os.getenv("MLFLOW_TRACKING_USERNAME")
os.environ['MLFLOW_TRACKING_PASSWORD'] = os.getenv("MLFLOW_TRACKING_PASSWORD")
import pandas as pd
import sys
from pathlib import Path 
sys.path.insert(0, str(Path.cwd().parent / "scripts"))  # if notebook is in a subfolder
# OR if the notebook is at the project root:
sys.path.insert(0, str(Path.cwd() / "scripts"))

In [ ]:
import mlflow

mlflow.set_experiment('electricity-distribution-forecast')


2026/07/30 15:06:45 INFO mlflow.tracking.fluent: Experiment with name 'electricity-distribution-forecast' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/7f3415414c0a400c827e6977c0ec79cc', creation_time=1785420407673, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1785420407673, lifecycle_stage='active', name='electricity-distribution-forecast', tags={}, trace_location=None, workspace='default'>

#### 1. Load raw joined dataset
Load pack (GridStatus load pull) + weather (Open-Meteo population-weighted) - already joined by the existing pull/concat scripts.

In [7]:
#load the joined raw dataframe (from DVC-tracked storage / local cache)
df_raw = pd.read_csv("../data/raw/ercot_south_central_raw.csv")


#### 2. Validate raw data with Great Expectations


In [10]:
from validate_raw_data import validate_dataframe
# df is the DataFrame you already loaded
passed = validate_dataframe(df_raw)
if not passed:
    raise RuntimeError("Data failed validation — fix issues before proceeding.")

2026-07-30 15:42:02  INFO      ================================================================
2026-07-30 15:42:02  INFO      GX Validation Gate  |  in-memory DataFrame  (92687 rows × 5 cols)
2026-07-30 15:42:02  INFO      ================================================================
2026-07-30 15:42:02  INFO      Source column names: ['timestamp', 'actual_load_mw', 'temp_c', 'humidity_pct', 'precip_mm']
2026-07-30 15:42:02  INFO      Columns renamed to canonical names: ['timestamp', 'actual_load_mw', 'temp_c', 'humidity_pct', 'precip_mm']
2026-07-30 15:42:02  INFO      Created temporary directory 'C:\Users\hp\AppData\Local\Temp\tmpt9o54jlc' for ephemeral docs site
2026-07-30 15:42:02  INFO      Loading 'datasources' ->
[]
2026-07-30 15:42:02  INFO      ExpectationSuite 'raw_data_validation_suite' registered (13 expectations).
2026-07-30 15:42:02  INFO      Running checkpoint …
Calculating Metrics: 100%|██████████| 55/55 [00:00<00:00, 134.22it/s]
2026-07-30 15:42:03  INFO      ====

#### 3. Baseline feature set
Calendar features (`hour`, `dayofweek`, `month`) + weather (`tavg`, `tmin`, `tmax`, `prcp`) + lagged load (`load_lag_24`, `load_lag_168`). Shared starting point for all three tree/linear models before any SHAP-guided engineering.

#### Experiment 1 - Linear Regression

### 1a. Baseline Linear Regression
Target: reproduce paper's baseline MAPE approx 5.37%

In [ ]:
with mlflow.start_run(run_name='lr_baseline'):
    # TODO: fit LinearRegression on df_baseline
    # TODO: mlflow.log_params({...})
    # TODO: mlflow.log_metrics({'mape': ..., 'rmse': ..., 'mae': ..., 'peak_mape': ...})
    # TODO: mlflow.sklearn.log_model(model, 'model')
    pass


### 1b. SHAP diagnosis on baseline LR
Expect `load_lag_24`, `load_roll_mean_24`, `load_lag_168` to dominate; static calendar features to contribute little.

In [ ]:
# TODO: import shap; run SHAP explainer on the baseline LR model
# TODO: log SHAP summary plot as an MLflow artifact


### 1c. Engineer SHAP-guided features for LR
Add: `CDD_lag_24`, `HDD_lag_24`, `temp_spike_vs_mean`, `is_extreme_cold_event`, `is_extreme_heat_event`, `hour_sin`, `dayofweek_cos`, `lag_24_x_hour`, `CDD_x_hour`

In [ ]:
# TODO: build_lr_engineered_features(df_baseline) -> df_lr_v2


### 1d. Retrain improved LR
Target: MAPE approx 4.74% (~11% relative improvement)

In [ ]:
with mlflow.start_run(run_name='lr_shap_engineered'):
    # TODO: fit + log, same pattern as 1a
    pass


---
## Experiment 2 - XGBoost (paper's best performer)

### 2a. Baseline XGBoost
Target: MAPE approx 3.15% (your earlier reproduction landed 3.35% val - close enough to validate the pipeline; revisit the val to test gap noted earlier if it persists here)

In [ ]:
with mlflow.start_run(run_name='xgb_baseline'):
    # TODO: fit XGBRegressor on df_baseline
    # TODO: log params/metrics/model as above
    pass


### 2b. SHAP diagnosis on baseline XGBoost
Expect underprediction during peak events (early-morning winter, late-afternoon summer); `month`/`dayofweek` dominating importance despite being static.

In [ ]:
# TODO: SHAP explainer + summary plot, logged as MLflow artifact


### 2c. Engineer SHAP-guided features for XGBoost
Add:
- load_spike_vs_mean = (load - load_roll_mean_24) / (load_roll_mean_24 + 1)
- temp_spike_vs_mean = (tmax - tavg) / (tavg + 1)
- tmax_roll_max_72
- CDD_x_hour = CDD x hour
- lag_24_x_hour = load_lag_24 x hour
- is_extreme_heat_event = 1[tmax > 95th percentile]
- is_monday

In [ ]:
# TODO: build_xgb_engineered_features(df_baseline) -> df_xgb_v2


### 2d. Retrain improved XGBoost
Target: MAPE approx 0.79% - the paper's headline result

In [ ]:
with mlflow.start_run(run_name='xgb_shap_engineered'):
    # TODO: fit + log
    pass


---
## Experiment 3 - LightGBM

### 3a. Baseline LightGBM
Target: MAPE approx 5.26%

In [ ]:
with mlflow.start_run(run_name='lgbm_baseline'):
    # TODO: fit LGBMRegressor on df_baseline
    pass


### 3b. SHAP diagnosis on baseline LightGBM

In [ ]:
# TODO: SHAP explainer + summary plot


### 3c. Engineer SHAP-guided features for LightGBM
**Note the formulas differ from XGBoost's version of the same-named features:**
- load_spike_vs_mean = (load - load_roll_mean_168) / (load_roll_std_168 + eps) (168h window, not 24h)
- temp_spike_vs_mean = tmax - tavg (unnormalized here)
- lag_24_x_hour = load_lag_24 x hour
- is_extreme_heat_event = 1[tmax > 95th percentile]

In [ ]:
# TODO: build_lgbm_engineered_features(df_baseline) -> df_lgbm_v2


### 3d. Retrain improved LightGBM
Target: MAPE approx 0.9-1.0% (paper's text and table disagree slightly: 0.98% vs 0.91% - don't chase an exact match)

In [ ]:
with mlflow.start_run(run_name='lgbm_shap_engineered'):
    # TODO: fit + log
    pass


---
## Experiment 4 & 5 - LSTM / BiLSTM (manual benchmark, SHAP not applied)
Kept outside the SHAP loop and outside automated CI/CD promotion - logged to MLflow for comparison only, tagged clearly as a benchmark run.

In [ ]:
with mlflow.start_run(run_name='lstm_benchmark'):
    mlflow.set_tag('role', 'manual_benchmark')
    # TODO: define + train LSTM (architecture is your own design choice - the paper doesn't publish layer count/hidden size/window length)
    # TODO: log params/metrics/model
    pass


In [ ]:
with mlflow.start_run(run_name='bilstm_benchmark'):
    mlflow.set_tag('role', 'manual_benchmark')
    # TODO: define + train BiLSTM
    pass


---
## 6. Compare all runs
Pull metrics across all logged runs in this experiment for a side-by-side comparison table.

In [ ]:
# TODO: mlflow.search_runs(experiment_names=['shef-day-ahead']) -> comparison dataframe


## 7. Register best model + set champion alias
Once you've picked a winner (gated on Peak-MAPE per the project's promotion rule), register it and set the `champion` alias - this is metadata-only in Postgres, the R2 artifact doesn't move.

In [ ]:
# TODO: mlflow.register_model(model_uri, 'shef-day-ahead-model')
# TODO: client.set_registered_model_alias('shef-day-ahead-model', 'champion', version)
